# OC6 Phase III — validation of aerodynamic loading under large platform motion

This notebook reproduces the figures of

> Bergua, R., Robertson, A., Jonkman, J., Branlard, E., Fontanella, A., Belloli, M., *et al.* (2023).
> **OC6 project Phase III: validation of the aerodynamic loading on a wind turbine rotor undergoing
> large motion caused by a floating support structure.** *Wind Energy Science* **8**, 465–485.
> <https://doi.org/10.5194/wes-8-465-2023>

from the complete campaign, and doubles as a template for **benchmarking your own aerodynamics code**
against it. The data is one record:

> Marykovskiy, Y., Bergua, R. (2026). **Floating Wind Turbine Rotor Aerodynamics Benchmark (IEA Wind
> Task 30 OC6 Phase III).** Zenodo. <https://doi.org/10.5281/zenodo.23593300>

| | in the record | from |
|---|---|---|
| **Simulations** | 56 numerical models, 11 load cases: hub loads, blade sections, hot-wire wake, PIV planes | Wind Data Hub [10.21947/1922900](https://doi.org/10.21947/1922900), via its Zenodo mirror [10.5281/zenodo.23572116](https://doi.org/10.5281/zenodo.23572116) |
| **Experiment 1** | the UNAFLOW campaign: tower loads, hot-wire wake and PIV, surge motion | [10.5281/zenodo.4740005](https://doi.org/10.5281/zenodo.4740005) |
| **Experiment 2** | the 2021 follow-on campaign: tower-top loads, surge *and* platform-pitch motion | Wind Data Hub, as above |

## Setup

Install the package once with `pip install -e ".[notebook]"` from the repository root. On RenkuLab a
session launcher built from this repository (`environment.yml`) has the dependencies already, and
the cell below imports the package straight from `src/`.

`oc6_phase3` finds its data in one of three places, in this order:

1. `$OC6_DATA_DIR`, if you set it;
2. a **RenkuLab data connector** for [10.5281/zenodo.23593300](https://doi.org/10.5281/zenodo.23593300)
   — the record is mounted as a folder next to this repository;
3. `./data/`, filled by `oc6.paths.data_dir(download=True)` (~2.2 GB from Zenodo).

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import oc6_phase3 as oc6
except ModuleNotFoundError:  # not installed, e.g. on RenkuLab: use the package of this repository
    sys.path.insert(0, str(Path.cwd().parent / "src"))
    import oc6_phase3 as oc6
from oc6_phase3 import (
    contribute, data, figures, harmonics, plots, registry, style, wake,
)

style.use()                 # project Plotly template: colour by modelling approach
pd.set_option("display.width", 140)

# style.use() also gives every figure an HTML copy, drawn where JupyterLab lacks Plotly's own
# renderer (on RenkuLab it runs in an environment of its own). It needs internet access, to load
# plotly.js. VS Code and Colab keep the renderer Plotly picks for them.

print("data directory:", oc6.paths.data_dir())
data.describe()

## What is in the dataset

The record has two layers, told apart by the file names. `oc6.phase3.archive.*` holds the source
records value for value — every row each participant submitted, every UNAFLOW test, the Experiment 2
measurements. `oc6.phase3.derived.*` holds what the paper calculates from them. `oc6_phase3.data`
reads both and adds, on read, what is looked up rather than stored: the participant label, the
modelling approach and the canonical motion phase.

**Modelling approach.** Every submission is named `oc6.phase3.<participant>.m<XY>.lc<NM>.<kind>.txt`,
where `X` is the wake/induction model (1 BEM, 2 dynamic BEM, 3 generalized dynamic wake, 4 free
vortex wake, 5 CFD) and `Y` is the blade/airfoil model (1 disk, 2 static polars, 3 unsteady airfoil
aerodynamics, 4 blade resolved). Those two digits are the axis the whole paper is organised along, so
they are resolved into columns here rather than left in the file name.

In [ ]:
models = data.models()
models.groupby(["wake_model", "airfoil_model"], observed=True).size().unstack(fill_value=0)

In [ ]:
models[["label", "organization", "country", "code", "model_code",
        "wake_model", "airfoil_model", "in_paper_table3"]]

**Load cases.** One steady condition, three surge frequencies, three equivalent platform-pitch
frequencies, and three verification cases with no measurements.

The pitch amplitudes are prescribed in degrees. To compare them with the surge cases they are
converted to an equivalent hub displacement, `sin(pitch) × 1.4706 m`, where the distance comes from
the Experiment 2 geometry in Table 2 of the paper. Because the prescribed pitch amplitudes are
rounded to 0.1°, LC 3.5 and LC 3.7 are only *approximately* equivalent to LC 2.5 and LC 2.7.

In [ ]:
data.load_cases()[
    ["load_case", "motion", "frequency_hz", "amplitude", "amplitude_unit",
     "surge_equivalent_amplitude_m", "reduced_frequency", "rotor_speed_rpm",
     "blade_pitch_deg", "validation", "description"]
]

### How much of the campaign is here

All of it. The Wind Data Hub archive was re-uploaded in full in October 2026, and its owner confirmed
the 4,130 files as the complete set; the record converts every one. 55 of the 56 models report hub
loads, and not every model ran every load case.

**Two models are not in the paper.** TUD `m52` (CFD) and EDF2 `m42` (FVW) are missing from its
Table 3: they took part in the wake analysis of [Cioni et al. (2023)](https://doi.org/10.5194/wes-8-1659-2023).
EDF2 `m42` reported wake data only, but TUD also submitted loads, so it appears in the figures below.
It is kept, and marked `*` wherever models are named; the per-approach counts include it, which is
why CFD has one model more than the paper (8 against 7 in steady wind). For Load Case 2.5 the
counts match the legend of the paper's Fig. 16: 13 BEM, 20 dynamic BEM, 2 GDW, 12 FVW, and 6 CFD
against the paper's 5.

**One record is irregular.** CENER `m43` submitted eight motion periods for Load Case 2.5 instead of
one. The summary takes the harmonic at the motion frequency, and the figures fold the record onto
one period, so it counts like every other submission.

In [ ]:
forces = data.forces()
coverage = (
    forces.assign(model=forces.label.astype(str) + " " + forces.model_code.astype(str))
    .pivot_table(index="model", columns="load_case", values="fx_n",
                 aggfunc="size", observed=True)
    .notna().astype(int)
)
print(f"{coverage.to_numpy().sum()} of {coverage.size} model x load-case combinations "
      f"have hub loads ({100 * coverage.to_numpy().mean():.0f} %)")
coverage.sum().rename("models with hub loads").to_frame().T

### The two experiments

Both campaigns ran the same 1:75 rotor in the Politecnico di Milano wind tunnel, and the paper
compares the models against both.

* **Experiment 1** (UNAFLOW) moved the turbine in surge on two hydraulic actuators and also measured
  the wake. Its tower-top balance is reduced to thrust and torque in the build step,
  `oc6_phase3.build.experiments`.
* **Experiment 2** (2021) moved it with a six-degree-of-freedom robot, in surge *and* platform
  pitch, so it is the only measurement for the pitch load cases. Its loads are reduced on read: the
  published channels are used as they are (thrust `Fx`, torque `Mx`), low-pass filtered at 3 Hz and
  phase-averaged like Experiment 1, and the amplitudes are normalised by the motion the robot
  actually achieved. That reproduces every Experiment 2 value the paper plots (Figs. 3, 4 and 16–18) to within about 2 %.

Every figure below draws Experiment 1 solid and Experiment 2 dashed, where they exist.

In [ ]:
data.experiment_summary()[
    ["experiment", "load_case", "fx_mean_n", "mx_mean_nm", "fx_peak_to_peak_n",
     "fx_phase_deg", "mx_phase_deg", "fx_peak_to_peak_per_amplitude_n_per_m"]
].round(3)

---

## Working in this notebook

Two things are worth knowing before the figures start, because they are what make this a
notebook you can work in rather than only read.

**Every figure takes its data as the first argument.** `figures.steady_loads()` draws the
packaged table; `figures.steady_loads(my_frame)` draws yours. Filter it, extend it, replace
it — the figure draws what it is handed. Pass `measured=False` to any of them to drop the
experimental reference, or a frame of your own measurements to replace it.

**Every figure is a short recipe over `oc6_phase3.plots`**, which holds the five drawing
moves they are all made of: `markers` and `lines` (one mark per model, coloured by wake
model and shaped by airfoil model), `measured` and `reference_line` (neutral ink, because
measurements are the reference rather than another series), and
`quartiles_per_approach` / `lines_per_approach` for the aggregates.

Here is Fig. 3 built from those primitives:

In [ ]:
frame = plots.sort_models(data.summary("1.1").dropna(subset=["fx_mean_n"]))
measured = data.experiment_summary()
measured = measured[measured.load_case == "1.1"]

fig = plots.figure("Aerodynamic rotor thrust, steady wind", "built inline",
                   height=480, margin=dict(b=130),
                   legend=dict(orientation="h", y=-0.4, x=0))
plots.markers(fig, frame, x="model", y="fx_mean_n", unit="N", value_label="thrust")
for experiment, row in measured.groupby("experiment", observed=True):
    plots.reference_line(fig, row.fx_mean_n.iloc[0], name=str(experiment).split(" (")[0])
fig.update_layout(yaxis_title="Thrust [N]")
fig.update_xaxes(tickangle=-60)
fig

Swap `markers` for `lines`, change the column, add a trace of your own — it is ordinary
Plotly underneath, and `fig` is a `go.Figure` you can keep editing. The legend names the
modelling approaches, not the fifty models: click an approach to hide or show all its models,
double-click it to show that approach alone. Hover over a line or marker to see which model it is.

When you want to see what a packaged figure does, read it:

In [ ]:
import inspect

print(inspect.getsource(figures.steady_quartiles))

### Choosing which models to draw

A figure draws whatever it is handed, so choosing models is ordinary pandas filtering before the
call. Every table carries the modelling approach (`wake_model`: BEM, DBEM, GDW, FVW, CFD), the
blade/airfoil model (`airfoil_model`), the participant (`label`) and the `model_code`. Only the
CFD models, in steady wind:

In [ ]:
summary = data.summary("1.1")
figures.steady_loads(summary[summary.wake_model == "CFD"])

The same filter works on the time series — the CFD models through one platform period of Load
Case 2.5:

In [ ]:
forces = data.forces("2.5")
figures.thrust_over_period(forces[forces.wake_model == "CFD"])

Other selections are one line each, on any table:

```python
summary[summary.wake_model.isin(["FVW", "CFD"])]                     # the higher-fidelity models
summary[summary.airfoil_model == "Unsteady airfoil aerodynamics"]    # one blade/airfoil model
summary[summary.label.isin(["NREL1", "NREL2"])]                      # one organisation's models
summary[~plots.not_in_paper(summary)]                                # exactly the paper's model set
```

`data.models()` lists all 56 with their code, approach and organisation, to pick from.

### Benchmarking your own model

This is what the dataset is for: run your code on the load cases the 56 models ran, put the results
through the same post-processing, and draw them into every figure below. Three steps.

**1. Set up the rotor and the load cases.** The specification is the OC6 Phase III definition
document: Robertson, Bergua, Fontanella and Jonkman (2023), NREL/TP-5000-83102,
[10.2172/1957538](https://doi.org/10.2172/1957538). Where to find each part, and where the record
has the same as a table:

| what | definition document | in this record |
|---|---|---|
| turbine: 2.38132 m rotor, 0.178 m hub, rigid blades and tower; the 5° shaft tilt is cancelled by a 5° tower pitch, so the rotor faces the wind squarely | Sect. 3.1, Tables 1–2, Fig. 3 | — |
| coordinate system of the outputs: CS3, origin at the hub centre | Fig. 3 | — |
| blade geometry: chord, twist, relative thickness and airfoil at 20 stations | Sect. 3.2, Table 3 | `data.blade_geometry()` |
| airfoil polars, 20 stations × 7 Reynolds numbers, as given to the participants | Sect. 3.2 (`airfoil_data.mat`) | `data.airfoil_polars()` |
| unsteady-airfoil parameters of the UNAFLOW AeroDyn airfoil files | — | `data.airfoil_unsteady_parameters()` |
| measured 2D polars of the SD7032 airfoil | — | `data.sd7032_polars()` |
| wind speed: the blockage-corrected U′∞, 4.19 m/s rather than the 4.00 m/s measured | Sect. 4, Table 5 | `data.load_cases().wind_speed_ms` |
| load cases: wind, rotor speed, blade pitch, motion, frequency and amplitude | Sect. 5, Tables 5–8 | `data.load_cases()` |
| outputs: columns, units, time step, sign conventions, file names | Sect. 6, Tables 9–14 | `data.forces()`, `data.blade_aero()`, `data.hotwire()`, `data.piv()` |
| how the participants set up each modelling approach | Appendix A.3 | `data.models()` |

The measured loads have the inertial loads removed, so the document gives no masses. If your code
needs them, use any value and remove the inertial loads from your outputs (Sects. 3.1, 3.3).

**2. Report as the participants did** (Sect. 6):

* hub loads `Fx … Mz` in CS3, without mass, inertia or gravity; thrust `Fx` is positive downwind;
* steady wind (LC 1.X): one line, averaged over one rotor revolution once the start-up transient
  has died out;
* platform motion (LC 2.X, 3.X): one full motion period after the transient, every 0.025 s. **Time
  zero is set by the motion**: surge at 0 m and moving upwind (a negative sine), platform pitch at
  0° and increasing (a positive sine). Every motion phase in this notebook is computed from that
  clock — NREL shifted the surge cases by 180° so that phase 0 is the platform at neutral, moving
  downwind, for both motions — so a record with another time zero lands at the wrong phase. Several
  periods on the same clock are fine; they are averaged onto one;
* blade sections at the nine output stations, 18.9 % to 98.3 % span
  (`data.blade_geometry().output_station`), averaged over the three blades; normal force positive
  downwind, tangential force positive in the direction of rotation.

This is what a submitted forces file holds — NREL1's BEM model in Load Case 2.5, the eight columns of
Table 10 in their order. The surge starts at 0 and decreases:

In [ ]:
forces = data.forces("2.5")
nrel = forces[(forces.label == "NREL1") & (forces.model_code == "m12")]
nrel[["time_s", "platform_displacement", "fx_n", "fy_n", "fz_n", "mx_nm", "my_nm", "mz_nm"]].head()

**3. Bring your results in**, one of two ways:

* **As output files.** Write the files Sect. 6 asks for — `MYCODE_M23_LC25_Forces.txt` (Table 10),
  `…_BA.txt` (Table 11), `…_HW_AW.txt` and `…_HW_CW.txt` (Table 12), `…_PIV_PSI0.txt` and
  `…_PIV_STEP1.txt` (Table 13) — and read a folder of them with `contribute.read_outputs`. It is the
  reader that converted the campaign's 4,122 submissions, so your files are held to the same layout
  rules. It returns `forces`, `blade_aero`, `hotwire` and `piv` with the campaign's columns, and
  `summary` with the validation metrics.
* **From memory.** For hub loads alone, `contribute.model_results` takes the time series as arrays.

Either way, the `model_code` in the name gives your model its colour and marker: `m23` is dynamic BEM
with unsteady airfoil aerodynamics, `m43` free vortex wake, `m54` blade-resolved CFD.

In [ ]:
# Replace this with your own results, either from output files:
#     mine = contribute.read_outputs("path/to/my_results", label="MyModel")
#     mine.summary, mine.forces, mine.blade_aero, mine.hotwire, mine.piv
# or, for hub loads alone, from arrays:
#     mine = contribute.model_results(
#         label="MyModel", model_code="m23", load_case="2.5",
#         time_s=t, fx_n=thrust, mx_nm=torque,
#     )
mine = contribute.example_model_results()
mine.summary[["label", "model_code", "wake_model", "load_case",
              "fx_mean_n", "fx_peak_to_peak_n", "fx_phase_deg"]].round(3)

In [ ]:
summary = pd.concat([data.summary("2.5"), mine.summary], ignore_index=True)
figures.amplitude_and_phase(summary, load_case="2.5")

In [ ]:
forces = pd.concat([data.forces("2.5"), mine.forces], ignore_index=True)
figures.thrust_over_period(forces, load_case="2.5")

The example model is synthetic and deliberately sits above the campaign so you can find
it. Every figure below accepts the same treatment.

---

## 1. Steady wind — does everyone model the same rotor?

Load Case 1.1 is the sanity check: 4.19 m s⁻¹ of uniform wind, 240 rpm, no motion, no tilt. The only
non-zero rotor loads are thrust and torque, and every model should land on the same two numbers.

All 55 models reported hub loads for this case, and 51 also reported blade aerodynamics. Those give
an independent check: integrating the sectional normal force over the blade and multiplying by three
blades gives the thrust, and the tangential force times radius gives the torque. The reported
stations stop at 18.9 % and 98.3 % span, so the integrand is extended linearly to zero at root and
tip. The integrated thrust agrees with the reported one to a median of 1 %. Three submissions differ
by more than 5 % — POLIMI, UoS and PRI — while the same integration agrees for everyone else, so the
difference lies in those submissions' two files rather than in the method.

In [ ]:
steady = data.summary("1.1")
both = steady.dropna(subset=["fx_mean_n", "fx_from_blade_aero_n"])
(
    both.assign(
        thrust_ratio=both.fx_from_blade_aero_n / both.fx_mean_n,
        torque_ratio=both.mx_from_blade_aero_nm / both.mx_mean_nm,
    )[["label", "model_code", "fx_mean_n", "fx_from_blade_aero_n", "thrust_ratio",
       "mx_mean_nm", "mx_from_blade_aero_nm", "torque_ratio", "blade_aero_sign_corrected"]]
    .round(3)
)

In [ ]:
figures.steady_loads(quantity="thrust")

In [ ]:
figures.steady_loads(quantity="torque")

Two things the paper draws out of these, and both hold for the complete set:

* **The free-vortex-wake and CFD models predict more thrust than the BEM models** — a median of
  35.8 N against 34.9 N — even though they use the same airfoil polars. Their axial induction is
  lower and their tangential induction higher; both raise the local inflow velocity, and the lower
  axial induction also raises the angle of attack.
* **Most models underpredict the torque**: 52 of the 55 sit below Experiment 1, and the two
  experiments agree on it to within 3 %.

The experiments disagree on thrust, though: Experiment 2 measured 33.7 N, below every model, and
Experiment 1 about 36 N. The paper attributes the difference to the cable bundle behind the turbine in
Experiment 2, or a small blade pitch offset.

Figure 5 summarises the same data per approach, as in the paper: the median, the interquartile range (thick bar) and the full range (whiskers). The CFD count includes TUD*.

In [ ]:
figures.steady_quartiles()

## 2. Steady wind — the wake

Experiment 1 traversed a hot-wire probe through the wake, along the wind direction and across it,
and recorded PIV planes behind the rotor; Experiment 2 has no wake measurements in the record. Only
participants using free vortex wake or CFD could report these — 17 models on the hot-wire traverses.

In [ ]:
figures.hotwire_alongwind()

The along-wind profile shows where the wake starts to recover: the station at which the velocity is a
minimum. The paper reads this **inflection wake recovery point** off the CFD models only — the FVW
wakes carry no viscous diffusion, so where they recover depends on the wake length each participant
chose. As in the paper, CENER lands near 4.2 m and DTU1, POLIMI and UNIFI at the last station or
beyond. The measured minimum is very flat: within 0.005 m/s between 3.50 and 3.83 m, so this record
puts it at 3.50 m where the paper reads 3.83 m.

In [ ]:
figures.wake_recovery_point()

In [ ]:
figures.hotwire_crosswind()

The crosswind profile is the more instructive of the two. The measured velocity drops to
1.75–2.75 m s⁻¹ inside the rotor region, and the models reproduce that. Where they part company is at
`y = 0`: 12 of the 17 models return a wind speed *above* the free stream directly behind the hub,
because they do not model the hub nose, while the experiment shows a clear deficit there.

That is exactly why the paper's average-deficit metric (Eq. 4) weights each point by its radius —
the disagreement at the centre is then worth almost nothing.

In [ ]:
figures.wake_deficit()

### Blade-tip vortices

The PIV plane sits just behind and above the rotor, and catches the tip vortices as they convect
downwind. Their path traces the wake expansion.

Following the paper, the vortex centres are found as local maxima of Graftieaux's **Γ₁** scalar,
which averages the sine of the angle between the position vector and the local velocity over a
neighbourhood; it tends to ±1 at a vortex core. Two details matter in practice and are worth naming,
because the published method does not specify them:

* the neighbourhood-mean velocity is subtracted first. Plain Γ₁ is not Galilean invariant, and a wake
  convecting at 4 m s⁻¹ drowns out the rotation — without this step the detector finds nothing;
* candidates must also be strong in vorticity and clear of the window edge, which is what keeps the
  detector from locking onto the shear layer.

Each centre is then moved onto the vorticity peak within 0.05 m of it: Γ₁ marks the vortex, the
vorticity peak locates its core. For the sharp measured vortices the two coincide; for the diffuse
vortices of some CFD models the Γ₁ maximum sits up to 0.04 m off the core.

The trajectory then keeps, per model, the dominant track: the centres within 12 mm of the straight,
rising line that most of them lie on. Some planes also show a second structure 0.1–0.15 m lower,
and a centre on the very edge of the search area may be the clipped flank of a peak outside it;
both are left out.

The circles below are the detected centres; compare them with Fig. 9b of the paper.

In [ ]:
figures.piv_vorticity(source="experiment", azimuth_deg=30.0)

In [ ]:
figures.tip_vortex_trajectory()

The measured trajectory rises from z ≈ 1.24 m to ≈ 1.27 m across the window. The shaded band holds
the measured centres themselves: they scatter by about ±0.01 m, one or two points of the 5 mm PIV
grid. The free-vortex-wake models sit above it and rise faster: they **overpredict the wake
expansion**. The paper's explanation is the wind tunnel ceiling, half a rotor diameter above the
blade tip, inhibiting expansion in the experiment — and indeed the CFD models that meshed the tunnel
walls, floor and ceiling, POLIMI and UNIFI, lie within about 0.01 m of the measurement, while DTU1's
CFD model, without the ceiling, expands like the FVW models. TUD `m52`*, which is not in the paper,
also sits with the measurement. CENER `m54` is missing here, as in the paper: the detector finds too
few of its vortices.

### How far to trust a single curve

This figure does not reproduce the paper's line for line. Read off the paper's Fig. 10, most curves
here lie within 0.01 m of the published ones, and a few (DTU1, ON, UPC) up to 0.02 m higher. That is
to be expected: the paper names the detector, Γ₁, but not its settings, and where a vortex centre
lands on a 5 mm grid depends on them. So rather than tune the detector until it matches the published
figure, the record states how much each curve depends on it. Every trajectory was extracted twelve
times: Γ₁ threshold 0.8, 0.85 or 0.9; vorticity above the 90th or the 95th percentile of the plane;
centre moved onto the vorticity peak or not. Below, each trajectory is evaluated near both ends of the
window, from a straight line through its track. The marker is the default detector, the one behind
Fig. 10; the bar spans the twelve settings.

In [ ]:
figures.tip_vortex_uncertainty()

For most models the bar is shorter than 0.01 m, and the measured trajectory moves by 0.002 m at
most. Four models depend on the detector:

* **UNIFI `m52`**, by 0.04 m. Its CFD vortices are diffuse and the Γ₁ maximum sits off the vorticity
  core; only with the centre moved onto the core does UNIFI sit with the measurement, as in the paper.
* **TUB `m43`**, by 0.01 m upstream and 0.06 m at the downstream end: the slope of its track changes.
* **TUD `m52`***, by 0.02 m downstream.
* **POLIMI `m52`** is found in only 2 of the 12 settings, the most permissive ones; the others find too
  few of its vortices for a trajectory.

None of this touches what the figure shows. At every setting, every free-vortex-wake model sits at
least 0.03 m above the measurement — three times the scatter of the measured centres. Read Fig. 10 for
that pattern, and for which models sit with the measurement, not for the order of models a few
millimetres apart.

In [ ]:
spread = data.tip_vortex_uncertainty()
spread["model"] = (spread.participant.astype(str).str.upper() + " " + spread.model_code.astype(str)
                   ).where(spread.source == "simulation", spread.source)
(spread.pivot_table(index="model", columns="x_m", aggfunc="first",
                    values=["z_default_m", "spread_m", "settings_found"])
       .sort_values(("spread_m", 1.2), ascending=False)
       .round(3))

The numbers behind the bars are a table of the record, `derived.*.tip_vortex_uncertainty`.
`oc6_phase3.build.derive.trajectory_sensitivity(data.piv("1.1"))` recomputes them in about
2.5 minutes, and takes a `settings` list if you want to try a detector of your own.

---

## 3. Unsteady wind — fixing the phase convention

Everything from here on is plotted against **platform motion phase**, and it is worth being explicit
about what that means, because the raw files are not consistent about it.

The definition document asks for surge to start as a *negative* sine and platform pitch as a
*positive* one, so the two motion types start in opposite directions. NREL resolved this by shifting
the surge results 180°. Here the same thing is done once, when `oc6_phase3.data` reads the tables, by
defining a single canonical phase (`harmonics.canonical_phase_deg`):

> **phase 0°** — platform at neutral, moving **downwind**. The rotor apparent wind, and hence the
> thrust, is at its **minimum**.
> **phase 180°** — platform at neutral, moving **upwind**; apparent wind and thrust at maximum.

A quasi-steady rotor follows the apparent wind exactly, so its thrust lags the platform displacement
by 90°. That number, 90°, is the reference for every phase shift below. For the measurements the
phase is recovered from the measured displacement.

In [ ]:
figures.platform_motion(load_case="2.5")

## 4. Unsteady wind — Load Case 2.5

1 Hz surge, ±35 mm. This is the case with the largest apparent-wind variation, so the measured
signal-to-noise ratio is the best of the campaign, and both experiments ran it.

Both measured curves are derived from the raw 2 kHz records. Experiment 1: thrust `= −TTFx` from the
UNAFLOW tower-top balance. Experiment 2: the published `Fx`. Each keeps only the frequencies below
3 Hz — a low-pass in the frequency domain, so without a phase lag — which removes the 4 Hz 1P response
of a ~10 % mass imbalance in one blade and, in the Experiment 1 record, a third harmonic of the motion
at exactly 3 Hz. Then it is phase-averaged over all motion periods.

In [ ]:
figures.thrust_over_period(load_case="2.5")

Every curve is a clean first-order sine: the response is driven by the platform motion alone. The
models sit between the two experiments in mean value (a median of 35.1 N, against 36.2 N and 34.2 N),
as they did in steady wind, and the spread in amplitude is small.

Figure 14 puts numbers on that. The peak-to-peak amplitude is twice the FFT amplitude at the motion
frequency; the phase shift comes from the real and imaginary parts of the same complex coefficient.
Experiment 2's thrust lags by 87°, slightly *less* than 90°, which the paper attributes to small
rotor speed oscillations in that campaign.

In [ ]:
figures.amplitude_and_phase(load_case="2.5", quantity="thrust")

Read the right-hand panel by marker shape:

* models using **static polars** (circles) land on 90° — a median of 90.1° — as a quasi-steady model
  must;
* models using **unsteady airfoil aerodynamics** (diamonds) mostly land slightly above, a median of
  90.8°.

That extra degree is not dynamic stall — the motion amplitudes were chosen to keep stall confined to
the blade root. It is the small hysteresis in attached-flow airfoil performance, and you can see it
directly by plotting the lift coefficient against angle of attack over one period.

In [ ]:
figures.lift_hysteresis(load_case="2.5", station=7)

These are NREL's two lifting-line models, identical except for the airfoil treatment — the pair of the
paper's Fig. 15. A static polar gives one lift coefficient per angle of attack, so its trace is a line
and the response is symmetric about the 180° phase. Unsteady airfoil aerodynamics opens the line into a
loop: lift is lower while the platform moves downwind than while it moves upwind, the symmetry breaks,
and the thrust phase shift rises above 90°.

The loop in the archive is narrower than the one the paper draws (ΔCl ≈ 0.01 against ≈ 0.02), and it
sits slightly *below* the static polar — by 0.002 on average, in every lifting-line pair in the
archive — so the static line runs along its upper edge rather than its middle. With ten times the
platform amplitude, in Load Case 2.12, the loop is wide enough for that offset to disappear:

In [ ]:
figures.lift_hysteresis(load_case="2.12", station=7)

Figure 16 collapses the individual models to one median per approach. Each model is first interpolated
onto a common phase grid — they report at different phases, and a median at the raw phases would mix a
different subset of models at every point.

In [ ]:
figures.approach_medians_over_period(load_case="2.5")

## 5. Across all the motion cases

Normalising the peak-to-peak load by the platform motion amplitude in metres makes surge and pitch
load cases directly comparable, and answers the central question of the project. Experiment 2 adds
the measured pitch cases; its points are normalised by the amplitude the robot achieved, which fell
short of the prescription at 2 Hz in pitch (0.26° of 0.3°).

In [ ]:
figures.normalised_amplitude_vs_frequency(quantity="thrust")

In [ ]:
figures.normalised_amplitude_vs_frequency(quantity="torque")

**The simulation points lie on a straight line through the origin.** The rotor load variation is
proportional to the change in apparent wind, at every frequency tested — the response is
quasi-steady. LC 2.12 is the cleanest demonstration: it has the same frequency as LC 2.7 but ten
times the amplitude, and once normalised it returns the same value (256 N/m against 259 N/m).

Surge and its equivalent pitch case give the same answer — 129 N/m for LC 3.5 against 130 N/m for
LC 2.5 — and the median phase stays within 1.5° of 90° throughout.

The measurements agree well at 0.125 and 1 Hz and scatter at 2 Hz, where the loads are smallest. The
paper notes a spurious 2.5 Hz component in the Experiment 1 torque at 2 Hz, which is why that point
sits so high in the torque figure; Experiment 2 does not show it.

For reference, these model-scale conditions correspond to full-scale periods of 12.5 s to 200 s and
nacelle motion amplitudes of 0.6 m to 9.4 m.

## 6. When the controller moves — the one case where fidelity matters

Load Cases 2.16 and 2.17 take LC 2.12's platform motion and add what a real turbine would do about
it: below rated, the rotor speeds up and slows down (±36 rpm about 240); above rated, the blade pitch
actuates (0° to 3°). There are no measurements for these — they are verification cases — but they are
where the modelling approaches finally separate.

In [ ]:
figures.amplitude_and_phase(load_case="2.16", quantity="thrust")

In [ ]:
figures.controller_cases()

With platform motion alone (LC 2.12) the BEM models land within 3 % of the approaches that model
dynamic inflow. Add rotor speed variation (LC 2.16) and BEM predicts a thrust variation about **9 %
smaller**; add blade pitch actuation (LC 2.17) and about **18 % larger**. The complete set
reproduces both numbers of the paper.

That is the practical conclusion of the project: the platform motion itself does not demand an
unsteady aerodynamic model, but a realistic operating condition with an active controller does.